# Debug OHLC Bar Order - Full Qubx Flow
# Uses a bare-minimum strategy that logs every bar update with out-of-order detection.
# Run and wait for minute boundaries to see if bars arrive out of order.

In [3]:
from pathlib import Path
from datetime import datetime, timezone

from qubx import logger
from qubx.core.basics import Signal, TriggerEvent
from qubx.core.interfaces import Instrument, IStrategy, IStrategyContext, IStrategyInitializer, MarketEvent
from qubx.utils.runner.runner import run_strategy_yaml


class DebugBarOrderStrategy(IStrategy):
    """Logs every market data bar with out-of-order detection."""

    live_base_subscription: str = "ohlc(1m)"
    live_event_schedule: str = "1min -1s"
    _last_bar_time: dict

    def on_init(self, initializer: IStrategyInitializer):
        self._last_bar_time = {}
        initializer.set_base_subscription(self.live_base_subscription)

    def on_start(self, ctx: IStrategyContext):
        logger.info(f"[DEBUG] Started with {len(ctx.instruments)} instruments")

    def on_warmup_finished(self, ctx: IStrategyContext):
        logger.info("[DEBUG] Warmup finished")
        if self.live_event_schedule:
            ctx.set_event_schedule(self.live_event_schedule)
        for i in ctx.instruments:
            ohlcv = ctx.ohlc(i)
            if ohlcv and ohlcv.times:
                t = datetime.fromtimestamp(ohlcv.times[0] / 1e9, tz=timezone.utc).strftime('%H:%M:%S')
                logger.info(f"[DEBUG] {i.symbol} OHLCV head: time={t} close={ohlcv.close[0]:.2f} len={len(ohlcv.times)}")

    def on_market_data(self, ctx: IStrategyContext, data: MarketEvent):
        if not hasattr(data.data, "time") or data.instrument is None:
            return None
        bar_time_ns = data.data.time
        sym = data.instrument.symbol
        prev = self._last_bar_time.get(sym)
        if prev is not None:
            if bar_time_ns < prev:
                bt = datetime.fromtimestamp(bar_time_ns / 1e9, tz=timezone.utc).strftime('%H:%M:%S.%f')[:-3]
                pt = datetime.fromtimestamp(prev / 1e9, tz=timezone.utc).strftime('%H:%M:%S.%f')[:-3]
                logger.warning(f"[DEBUG] {sym} OUT OF ORDER! bar={bt} prev={pt} dtype={data.type}")
            elif bar_time_ns > prev:
                bt = datetime.fromtimestamp(bar_time_ns / 1e9, tz=timezone.utc).strftime('%H:%M:%S')
                logger.info(f"[DEBUG] {sym} NEW CANDLE bar={bt} close={data.data.close:.2f} dtype={data.type}")
        self._last_bar_time[sym] = bar_time_ns
        return None

    def on_event(self, ctx: IStrategyContext, event: TriggerEvent):
        return None

In [4]:
# Override the strategy class in the loaded config and run
from qubx.utils.runner.runner import run_strategy, AccountConfigurationManager
from qubx.utils.runner.config import load_strategy_config_from_yaml
import qubx.connectors  # register ccxt connector

config_path = Path("configs/debug_ohlc_order.yaml")
stg_config = load_strategy_config_from_yaml(config_path)
stg_config.strategy = DebugBarOrderStrategy  # use class directly, not string

acc_manager = AccountConfigurationManager(None, config_path.parent, search_qubx_dir=True)

ctx = run_strategy(
    stg_config,
    acc_manager,
    paper=True,
    no_emission=True,
    no_notifiers=True,
    no_exporters=True,
)

ModuleNotFoundError: No module named 'qubx.utils.runner.config'

In [ ]:
# After interrupting, check the strategy log files in logs/ for details

In [ ]:
# cleanup cell - not needed

In [ ]:
# placeholder

In [ ]:
# placeholder